# Part 2: English to Hindi Translation using a Pre-trained Transformer Model

# Import Required Libraries
Install and import all necessary libraries for the translation evaluation task, including transformers, datasets, evaluate, nltk, and torch.



In [ ]:
# Install necessary libraries
%pip install datasets transformers sacrebleu rouge_score nltk torch sentencepiece evaluate matplotlib pandas seaborn

# Import required libraries
import torch
import nltk
import evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import MarianTokenizer, MarianMTModel
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
from IPython.display import display, HTML


# Download Required NLTK Resources
Download required NLTK resources such as punkt, wordnet, and punkt_tab for text processing and evaluation metrics.

In [ ]:
# Download necessary NLTK data
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("punkt_tab")

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!




## 1. Dataset Loading and Exploration



In [ ]:
# Load the IITB English-Hindi dataset
dataset = load_dataset("cfilt/iitb-english-hindi")

# Display dataset structure
print("\nDataset Information:")
print(dataset)

# Display dataset statistics
print("\nTraining set size:", len(dataset["train"]))
print("Validation set size:", len(dataset["validation"]))
print("Test set size:", len(dataset["test"]))

# Display a few examples from the test set
print("\nSample from test set:")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(f"English: {dataset['test'][i]['translation']['en']}")
    print(f"Hindi: {dataset['test'][i]['translation']['hi']}")


Dataset Information:
DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})

Training set size: 1659083
Validation set size: 520
Test set size: 2507

Sample from test set:

Example 1:
English: A black box in your car?
Hindi: आपकी कार में ब्लैक बॉक्स?

Example 2:
English: As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Hindi: जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।

Example 3:
English: The devices, which track every mile a motorist drives an



## 2. Initialize Pre-trained Model
Initialize the MarianMT tokenizer and model for English to Hindi translation from Helsinki-NLP, and move the model to the appropriate device (CPU/GPU).


In [ ]:
def setup_models():
    models = {}

    # Set up device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Model 1: Helsinki-NLP/opus-mt-en-hi (MarianMT)
    print("Loading Helsinki-NLP/opus-mt-en-hi...")
    models["marian"] = {
        "name": "Helsinki-NLP/opus-mt-en-hi",
        "tokenizer": MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi"),
        "model": MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-hi").to(device)
    }

    # Model 2: Facebook's mBART-large-50
    print("Loading facebook/mbart-large-50...")
    mbart_tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")
    mbart_tokenizer.src_lang = "en_XX"
    mbart_tokenizer.tgt_lang = "hi_IN"

    models["mbart"] = {
        "name": "facebook/mbart-large-50",
        "tokenizer": mbart_tokenizer,
        "model": MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50").to(device)
    }

    # Model 3: Google's MT5
    print("Loading google/mt5-small...")
    models["mt5"] = {
        "name": "google/mt5-small",
        "tokenizer": MT5Tokenizer.from_pretrained("google/mt5-small"),
        "model": MT5ForConditionalGeneration.from_pretrained("google/mt5-small").to(device)
    }

    return models, device

# Load all models
models, device = setup_models()

Using device: cuda
Loading Helsinki-NLP/opus-mt-en-hi...


/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading facebook/mbart-large-50...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


Loading google/mt5-small...




## 3. Define Translation and Evaluation Functions



In [ ]:
# Load evaluation metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# Function to compute evaluation metrics
def compute_metrics(predictions, references, model=None):
    # Compute BLEU score
    bleu_score = bleu_metric.compute(predictions=predictions, references=references)["bleu"]

    # Compute ROUGE scores with more details
    rouge_scores = rouge_metric.compute(
        predictions=predictions,
        references=[ref[0] for ref in references],
        rouge_types=["rouge1", "rouge2", "rougeL"]
    )

    rouge1 = rouge_scores["rouge1"]
    rouge2 = rouge_scores["rouge2"]
    rouge_l = rouge_scores["rougeL"]

    # Compute METEOR score
    from nltk.translate.meteor_score import meteor_score
    meteor_scores = []
    for pred, ref in zip(predictions, references):
        try:
            score = meteor_score([ref[0].split()], pred.split())
            meteor_scores.append(score)
        except Exception as e:
            # Handle potential errors in METEOR calculation
            meteor_scores.append(0)
    meteor_avg = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0

    # Calculate eval_loss if model is provided (will be None for pre-trained models)
    eval_loss = None
    if model is not None:
        # This is a placeholder. Actual loss calculation would depend on your model's interface
        # and tokenizer, which would need to be passed in or accessed globally
        eval_loss = 0.0  # Placeholder value

    return {
        "eval_BLEU": bleu_score,
        "eval_ROUGE-1": rouge1,
        "eval_ROUGE-2": rouge2,
        "eval_ROUGE-L": rouge_l,
        "eval_METEOR": meteor_avg,
        "eval_loss": eval_loss,
        # Keep the original keys for backward compatibility
        "BLEU": bleu_score,
        "ROUGE-L": rouge_l,
        "METEOR": meteor_avg
    }

In [ ]:
# Function to translate a single English sentence to Hindi using different models
def translate(english_text, model_key="marian"):
    model_info = models[model_key]
    model = model_info["model"]
    tokenizer = model_info["tokenizer"]

    # Different handling based on model architecture
    if model_key == "marian":
        # MarianMT model
        inputs = tokenizer(english_text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=128)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    elif model_key == "mbart":
        # mBART model - ensure we're using the right language codes
        tokenizer.src_lang = "en_XX"
        inputs = tokenizer(english_text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
                max_length=128
            )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    elif model_key == "mt5":
        # MT5 model - provide explicit translation instruction
        prefix = "translate English to Hindi: "
        inputs = tokenizer(prefix + english_text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=128)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Function to evaluate a specific model on the test dataset
def evaluate_model(test_data, model_key="marian", num_samples=None):
    predictions, references, english_texts = [], [], []
    num_samples = len(test_data) if num_samples is None else min(num_samples, len(test_data))

    model_info = models[model_key]
    model_name = model_info["name"]
    print(f"Evaluating {model_name} on {num_samples} examples...")

    for i, example in enumerate(test_data):
        if i >= num_samples:
            break
        if i % 50 == 0:
            print(f"Processing example {i}/{num_samples}")

        source_text = example["translation"]["en"]
        reference_text = example["translation"]["hi"]

        # Generate translation
        translation = translate(source_text, model_key)

        predictions.append(translation)
        references.append([reference_text])
        english_texts.append(source_text)

    # Compute metrics
    metrics = compute_metrics(predictions, references)

    return predictions, references, english_texts, metrics

# Function to evaluate all models
def evaluate_all_models(test_data, num_samples=None):
    results = {}
    all_english_texts = []
    all_references = []

    # First extract the English texts and reference Hindi translations
    num_samples = len(test_data) if num_samples is None else min(num_samples, len(test_data))
    print(f"Preparing to evaluate {len(models)} models on {num_samples} examples...")

    for i, example in enumerate(test_data):
        if i >= num_samples:
            break
        if i % 50 == 0:
            print(f"Processing example {i}/{num_samples}")

        all_english_texts.append(example["translation"]["en"])
        all_references.append([example["translation"]["hi"]])

    # Evaluate each model
    for model_key, model_info in models.items():
        print(f"\nEvaluating {model_info['name']}...")
        predictions = []

        # Generate translations
        for i, source_text in enumerate(all_english_texts):
            if i % 50 == 0:
                print(f"Translating example {i}/{num_samples} with {model_info['name']}")

            translation = translate(source_text, model_key)
            predictions.append(translation)

        # Compute metrics for this model
        metrics = compute_metrics(predictions, all_references)

        # Store results
        results[model_key] = {
            "name": model_info["name"],
            "predictions": predictions,
            "metrics": metrics
        }

    return results, all_english_texts, all_references



## 4. Trial Run on a Small Subset



In [ ]:
# Test the translation function with all models
sample_text = "The weather is nice today."
print(f"\nSample Translations:")
print(f"English: {sample_text}")
for model_key in models:
    print(f"{models[model_key]['name']}: {translate(sample_text, model_key)}")

# Evaluate on a small subset first to make sure everything works
small_test_data = dataset["test"].select(range(50))
print("\nRunning evaluation on 50 test examples for all models...")
eval_results, eng_small, refs_small = evaluate_all_models(small_test_data)

# Display results for all models
print("\nResults on small test set:")
for model_key, result in eval_results.items():
    print(f"\nModel: {result['name']}")
    for metric, value in result['metrics'].items():
        if value is None:
            print(f"{metric}: None")
        else:
            print(f"{metric}: {value:.4f}")


Sample Translations:
English: The weather is nice today.
Helsinki-NLP/opus-mt-en-hi: मौसम आज अच्छा है.
facebook/mbart-large-50: The weather is nice today.
google/mt5-small: <extra_id_0>

Running evaluation on 50 test examples for all models...
Preparing to evaluate 3 models on 50 examples...
Processing example 0/50

Evaluating Helsinki-NLP/opus-mt-en-hi...
Translating example 0/50 with Helsinki-NLP/opus-mt-en-hi

Evaluating facebook/mbart-large-50...
Translating example 0/50 with facebook/mbart-large-50

Evaluating google/mt5-small...
Translating example 0/50 with google/mt5-small

Results on small test set:

Model: Helsinki-NLP/opus-mt-en-hi
eval_BLEU: 0.0798
eval_ROUGE-1: 0.1280
eval_ROUGE-2: 0.0200
eval_ROUGE-L: 0.1280
eval_METEOR: 0.2878
eval_loss: None
BLEU: 0.0798
ROUGE-L: 0.1280
METEOR: 0.2878

Model: facebook/mbart-large-50
eval_BLEU: 0.0000
eval_ROUGE-1: 0.0231
eval_ROUGE-2: 0.0066
eval_ROUGE-L: 0.0219
eval_METEOR: 0.0038
eval_loss: None
BLEU: 0.0000
ROUGE-L: 0.0219
METEOR: 0



## 5. Full Model Evaluation



In [ ]:
# Evaluate on a larger subset of the test dataset (500 examples for comparison)
print("\nEvaluating on 500 test examples...")
test_subset = dataset["test"].select(range(500))
full_eval_results, eng_full, refs_full = evaluate_all_models(test_subset)

print("\nFinal Results:")
for model_key, result in full_eval_results.items():
    print(f"\nModel: {result['name']}")
    for metric, value in result['metrics'].items():
        if value is None:
            print(f"{metric}: None")
        else:
            print(f"{metric}: {value:.4f}")


Evaluating on 500 test examples...
Preparing to evaluate 3 models on 500 examples...
Processing example 0/500
Processing example 50/500
Processing example 100/500
Processing example 150/500
Processing example 200/500
Processing example 250/500
Processing example 300/500
Processing example 350/500
Processing example 400/500
Processing example 450/500

Evaluating Helsinki-NLP/opus-mt-en-hi...
Translating example 0/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 50/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 100/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 150/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 200/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 250/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 300/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 350/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 400/500 with Helsinki-NLP/opus-mt-en-hi
Translating example 450/500 with Helsinki-NLP/opus-mt-en-hi

Evaluat



## 6. Visualizing and Analyzing Results





## 7. Save Evaluation Results and Key Examples



In [ ]:
# Save metrics for all models
all_metrics = pd.DataFrame(columns=['Model'])

# Get all metric keys from the first model's results
metric_keys = list(next(iter(full_eval_results.values()))['metrics'].keys())

for model_key, result in full_eval_results.items():
    model_name = result['name']
    metrics = result['metrics']

    # Create a dictionary for this model's metrics
    model_data = {'Model': model_name}

    # Add each metric, handling None values
    for k, v in metrics.items():
        model_data[k] = v  # pandas handles None values appropriately in DataFrames

    # Append to the DataFrame
    all_metrics = pd.concat([all_metrics, pd.DataFrame([model_data])], ignore_index=True)

# Save metrics to CSV
all_metrics.to_csv('all_models_metrics.csv', index=False)

# Create a consolidated results dataframe
results_df = pd.DataFrame({
    'English': eng_full,
    'Reference': [ref[0] for ref in refs_full],
})

# Add translations from each model
for model_key, result in full_eval_results.items():
    results_df[f"{model_key}_translation"] = result['predictions']

# Save the full results DataFrame for later analysis
results_df.to_csv('all_translations_results.csv', index=False)

print("\nSaved evaluation results to CSV files.")

# Create a more concise comparison dataframe with just 10 examples
comparison_df = pd.DataFrame({
    'English': eng_full[:10],  # Take first 10 examples
    'Reference': [ref[0] for ref in refs_full[:10]]
})

# Add translations from each model for the first 10 examples
for model_key, result in full_eval_results.items():
    comparison_df[result['name']] = result['predictions'][:10]

# Save sample translations to CSV for easy comparison
comparison_df.to_csv('model_comparison_samples.csv', index=False)

print("Saved sample translations to model_comparison_samples.csv")


Saved evaluation results to CSV files.
Saved sample translations to model_comparison_samples.csv


## 8. Adding Remaining Visualisations and saving them



## 9. Summary and Next Steps for Part 3



In [ ]:
# Print summary for Part 2
print("\n" + "="*50)
print("PART 2 SUMMARY: Pre-trained Model Evaluation")
print("="*50)
print(f"Models evaluated:")
for model_key, model_info in models.items():
    print(f" - {model_info['name']}")

print(f"\nTest set size: {len(test_subset)} examples")
print("\nEvaluation Metrics:")

# Create a comparison table - handle None values properly
for model_key, result in full_eval_results.items():
    print(f"\n{result['name']}:")
    for metric, value in result['metrics'].items():
        if value is None:
            print(f"{metric}: None")
        else:
            print(f"{metric}: {value:.4f}")

# Determine best model - only consider metrics that aren't None
best_bleu_score = -1
best_model_name = None

for model_key, result in full_eval_results.items():
    bleu_score = result['metrics'].get('BLEU')
    if bleu_score is not None and bleu_score > best_bleu_score:
        best_bleu_score = bleu_score
        best_model_name = result['name']

if best_model_name:
    print(f"\nBest performing model based on BLEU score: {best_model_name}")
else:
    print("\nCould not determine best model (BLEU scores may be None)")

# Notes for Part 3 (Comparative Analysis)
print("\nNEXT STEPS (PART 3):")
print("1. Compare these results with the fine-tuned model from Part 1")
print("2. Analyze strengths and weaknesses of each approach")
print("3. Consider which approach is more suitable for different scenarios")
print("4. Document insights in the technical report")


PART 2 SUMMARY: Pre-trained Model Evaluation
Models evaluated:
 - Helsinki-NLP/opus-mt-en-hi
 - facebook/mbart-large-50
 - google/mt5-small

Test set size: 500 examples

Evaluation Metrics:

Helsinki-NLP/opus-mt-en-hi:
eval_BLEU: 0.1089
eval_ROUGE-1: 0.1128
eval_ROUGE-2: 0.0178
eval_ROUGE-L: 0.1112
eval_METEOR: 0.2837
eval_loss: None
BLEU: 0.1089
ROUGE-L: 0.1112
METEOR: 0.2837

facebook/mbart-large-50:
eval_BLEU: 0.0036
eval_ROUGE-1: 0.0357
eval_ROUGE-2: 0.0053
eval_ROUGE-L: 0.0324
eval_METEOR: 0.0058
eval_loss: None
BLEU: 0.0036
ROUGE-L: 0.0324
METEOR: 0.0058

google/mt5-small:
eval_BLEU: 0.0000
eval_ROUGE-1: 0.0019
eval_ROUGE-2: 0.0006
eval_ROUGE-L: 0.0019
eval_METEOR: 0.0000
eval_loss: None
BLEU: 0.0000
ROUGE-L: 0.0019
METEOR: 0.0000

Best performing model based on BLEU score: Helsinki-NLP/opus-mt-en-hi

NEXT STEPS (PART 3):
1. Compare these results with the fine-tuned model from Part 1
2. Analyze strengths and weaknesses of each approach
3. Consider which approach is more suitable

## 10. Saving the files locally

In [ ]:
# Simple code to download all files in the root directory
import os
from google.colab import files
import zipfile
from datetime import datetime

# Create a timestamped zip filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"all_files_{timestamp}.zip"

# Get all files in the current directory (excluding hidden files and directories)
all_files = [f for f in os.listdir('.') if os.path.isfile(f) and not f.startswith('.')]

print(f"Found {len(all_files)} files to zip:")
for file in all_files[:5]:  # Show first 5 files
    print(f" - {file}")
if len(all_files) > 5:
    print(f"...and {len(all_files) - 5} more files")

# Create the zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in all_files:
        zipf.write(file)

print(f"\nCreated zip file: {zip_filename}")

# Download the zip file
files.download(zip_filename)
print("Download initiated. Check your browser downloads.")

Found 18 files to zip:
 - bleu_distribution_mt5.png
 - comparison_bleu.png
 - model_comparison_samples.csv
 - bleu_distribution_mbart.png
 - translation_model_results_20250401_234758.zip
...and 13 more files

Created zip file: all_files_20250401_235247.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. Check your browser downloads.
